# Qwen2.5-VL: استخراج عنوان کتاب فارسی از تصویر جلد

نسخه اصلاح‌شده برای Google Colab. اصلاحات مهم:
- پاک‌سازی داده و بررسی GPU
- ارزیابی مدل پایه و مدل fine-tuned روی همان test set
- ماسک‌کردن prompt و padding در loss
- Decode کردن فقط توکن‌های تولیدشده، نه prompt
- محاسبه معیارها و نمودار از خروجی واقعی؛ بدون اعداد دستی
- ذخیره آداپتر و نتایج

**قبل از اجرا:** در Colab از `Runtime → Change runtime type` یک GPU انتخاب کن. ابتدا `SMOKE_TEST = True` را اجرا کن؛ برای اجرای کامل آن را `False` کن.

## 1. نصب وابستگی‌ها

این سلول را پس از انتخاب GPU اجرا کن. نسخه Transformers زیر نسخه 5 نگه داشته شده تا از تغییرات ناسازگار نسخه اصلی جلوگیری شود.

In [ ]:
%pip install -q -U "transformers>=4.49.0,<5.0.0" accelerate datasets peft bitsandbytes qwen-vl-utils python-Levenshtein matplotlib pandas pillow tqdm opencv-python-headless

## 2. تنظیمات و بررسی GPU

In [ ]:
import os, random, shutil, json, unicodedata
from pathlib import Path
from io import BytesIO
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from datasets import Image as HFImage
from Levenshtein import ratio as levenshtein_ratio
from transformers import (
    AutoProcessor, Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig, TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
DATASET_NAME = "shenasa/bookroom-persian-book-covers-and-titles"
SMOKE_TEST = True  # برای اجرای کامل: False
TRAIN_LIMIT = 32 if SMOKE_TEST else 1000
EVAL_LIMIT = 10 if SMOKE_TEST else 200
NUM_EPOCHS = 1
OUTPUT_DIR = "./qwen25vl-bookcover-final"
RESULTS_DIR = "./qwen25vl-evaluation-results"
PROMPT = "عنوان دقیق کتاب فارسی روی جلد را استخراج کن. فقط عنوان را بنویس."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("GPU پیدا نشد. در Colab از Runtime → Change runtime type یک GPU انتخاب کن.")

USE_BF16 = bool(torch.cuda.is_bf16_supported())
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", USE_BF16, "| Smoke test:", SMOKE_TEST)

## 3. بارگذاری دیتاست

In [ ]:
dataset = load_dataset(DATASET_NAME)
for split in ("train", "test"):
    if split not in dataset:
        raise ValueError(f"Split '{split}' در دیتاست پیدا نشد: {list(dataset.keys())}")
    missing = {"image", "text"} - set(dataset[split].column_names)
    if missing:
        raise ValueError(f"ستون‌های لازم پیدا نشدند: {missing}; ستون‌ها: {dataset[split].column_names}")

print(dataset)
print("Train:", len(dataset["train"]), "| Test:", len(dataset["test"]))

## 4. پاک‌سازی تصاویر و آماده‌سازی splitها

برای خواندن WebP از OpenCV استفاده می‌شود و تصویرهای خراب از دیتاست کنار گذاشته می‌شوند.

In [ ]:
# CELL 4 — Validate images with OpenCV and keep raw HF image records

from pathlib import Path
from PIL import Image
from datasets import Image as HFImage
import cv2
import numpy as np

def decode_image(image_item):
    """Decode an HF Image(decode=False) record into an RGB PIL image."""
    if isinstance(image_item, Image.Image):
        return image_item.convert("RGB")

    if isinstance(image_item, dict):
        raw = image_item.get("bytes")
        image_path = image_item.get("path")

        if raw is not None:
            if isinstance(raw, memoryview):
                raw = raw.tobytes()
            arr = np.frombuffer(raw, dtype=np.uint8)
            decoded = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            if decoded is not None:
                return Image.fromarray(cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB))

        if image_path and Path(image_path).is_file():
            decoded = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if decoded is not None:
                return Image.fromarray(cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB))

    if isinstance(image_item, str) and Path(image_item).is_file():
        decoded = cv2.imread(image_item, cv2.IMREAD_COLOR)
        if decoded is not None:
            return Image.fromarray(cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB))

    raise ValueError("Could not decode image using OpenCV. Check image bytes/path.")


def is_valid_image(example):
    try:
        image = decode_image(example.get("image"))
        image.load()
        return image.width > 0 and image.height > 0
    except Exception:
        return False


def prepare_split(split, limit, split_name, scan_limit, shuffle=False):
    # Keep raw bytes in the dataset; avoid HF/Pillow automatic decoding.
    raw = split.cast_column("image", HFImage(decode=False))
    if shuffle:
        raw = raw.shuffle(seed=SEED)

    raw = raw.select(range(min(scan_limit, len(raw))))
    print(f"Checking {len(raw)} raw images in {split_name}...")

    clean = raw.filter(
        is_valid_image,
        desc=f"Validating {split_name} images"
    )
    print(f"Valid images in {split_name}: {len(clean)}")

    if len(clean) == 0:
        first = raw[0].get("image") if len(raw) else None
        print("Raw image record type:", type(first).__name__)
        if isinstance(first, dict):
            raw_bytes = first.get("bytes")
            print("Path:", first.get("path"))
            print("Bytes length:", len(raw_bytes) if raw_bytes is not None else None)
            print("Header:", repr(raw_bytes[:16]) if raw_bytes is not None else None)
        raise ValueError(f"No valid images found in {split_name}.")

    clean = clean.select(range(min(limit, len(clean))))
    first = clean[0]
    image = decode_image(first["image"])

    if not str(first.get("text") or "").strip():
        raise ValueError(f"First retained sample in {split_name} has no title.")

    print(f"{split_name}: {len(clean)} usable samples; first image size={image.size}")
    return clean


train_scan = min(
    len(dataset["train"]),
    max(TRAIN_LIMIT * 5, TRAIN_LIMIT + 100)
)
test_scan = len(dataset["test"])

train_ds = prepare_split(
    dataset["train"], TRAIN_LIMIT, "train",
    scan_limit=train_scan, shuffle=True
)
test_ds = prepare_split(
    dataset["test"], EVAL_LIMIT, "test",
    scan_limit=test_scan, shuffle=False
)

print("Dataset preparation complete.")
print("Train:", len(train_ds))
print("Test :", len(test_ds))
print("Example title:", train_ds[0]["text"])

## 5. بارگذاری مدل پایه

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
# کاهش تعداد پیکسل‌ها برای کنترل مصرف حافظه در Colab
if hasattr(processor.image_processor, "min_pixels"):
    processor.image_processor.min_pixels = 256 * 28 * 28
if hasattr(processor.image_processor, "max_pixels"):
    processor.image_processor.max_pixels = 512 * 28 * 28

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", torch_dtype=COMPUTE_DTYPE
)
model.eval()
model.config.use_cache = True
print("Base model loaded.")

## 6. توابع inference و معیارها

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text or ""))
    text = text.replace("ي", "ی").replace("ك", "ک").replace("‌", " ")
    text = text.replace("\u0640", "")
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    text = text.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
    return " ".join(text.split()).strip().lower()

def user_messages():
    return [{"role":"user","content":[
        {"type":"image"},
        {"type":"text","text":PROMPT}
    ]}]

def generate_title(model_obj, image, max_new_tokens=40):
    # Accept a PIL image or raw HF Image(decode=False) record.
    image = decode_image(image).convert("RGB")
    prompt = processor.apply_chat_template(
        user_messages(), tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt], images=[image], padding=True, return_tensors="pt"
    )
    inputs = {k: v.to(model_obj.device) for k, v in inputs.items()}
    with torch.inference_mode():
        output_ids = model_obj.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(
        new_ids, skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0].strip()

def word_f1(pred, ref):
    p, r = normalize_text(pred).split(), normalize_text(ref).split()
    if not p or not r:
        return float(not p and not r)
    overlap = sum((Counter(p) & Counter(r)).values())
    precision, recall = overlap / len(p), overlap / len(r)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def calculate_metrics(predictions, references):
    exact, lev, wf1 = [], [], []
    for pred, ref in zip(predictions, references):
        p, r = normalize_text(pred), normalize_text(ref)
        exact.append(float(p == r))
        lev.append(levenshtein_ratio(p, r))
        wf1.append(word_f1(p, r))
    return {
        "Exact Match": float(np.mean(exact)),
        "Levenshtein Ratio": float(np.mean(lev)),
        "Word F1": float(np.mean(wf1)),
    }

def evaluate_model(model_obj, eval_ds, description):
    model_obj.eval()
    preds, refs, rows, failures = [], [], [], 0
    for i, ex in enumerate(tqdm(eval_ds, desc=description)):
        ref = str(ex["text"])
        try:
            pred = generate_title(model_obj, ex["image"])
        except Exception as err:
            print(f"Sample {i} failed: {type(err).__name__}: {err}")
            pred, failures = "", failures + 1
        preds.append(pred)
        refs.append(ref)
        rows.append({"index": i, "ground_truth": ref, "prediction": pred})
    metrics = calculate_metrics(preds, refs)
    metrics["Evaluation failures"] = failures
    return metrics, pd.DataFrame(rows)

## 7. ارزیابی مدل پایه

ارزیابی پایه قبل از اتصال LoRA انجام می‌شود؛ نتایج این سلول برای مقایسه نهایی نگه داشته می‌شوند.

In [ ]:
baseline_metrics, baseline_df = evaluate_model(model, test_ds, "Base model")
display(pd.DataFrame([baseline_metrics]))
display(baseline_df.head(10))

## 8. اتصال LoRA/QLoRA

In [ ]:
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Collator با ماسک‌کردن prompt و padding

در این سلول، loss فقط روی پاسخ دستیار محاسبه می‌شود.

In [ ]:
def collate_fn(examples):
    texts, prompt_texts, images = [], [], []
    for ex in examples:
        image = decode_image(ex["image"])
        answer = str(ex["text"]).strip()
        user = user_messages()
        full_messages = user + [{"role":"assistant","content":[{"type":"text","text":answer}]}]
        prompt_texts.append(processor.apply_chat_template(
            user, tokenize=False, add_generation_prompt=True
        ))
        texts.append(processor.apply_chat_template(
            full_messages, tokenize=False, add_generation_prompt=False
        ))
        images.append(image)

    batch = processor(text=texts, images=images, padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    # محاسبه طول prompt با همان تصویر؛ فقط پاسخ دستیار وارد loss می‌شود.
    for i, (prompt, image) in enumerate(zip(prompt_texts, images)):
        prompt_batch = processor(
            text=[prompt], images=[image], padding=False, return_tensors="pt"
        )
        prompt_len = prompt_batch["input_ids"].shape[1]
        labels[i, :min(prompt_len, labels.shape[1])] = -100

    batch["labels"] = labels
    return batch

check_batch = collate_fn([train_ds[0]])
n_target_tokens = int((check_batch["labels"] != -100).sum().item())
print("Collator test passed. Target tokens:", n_target_tokens)
if n_target_tokens == 0:
    raise ValueError("همه labelها ماسک شده‌اند؛ قالب prompt/answer باید بررسی شود.")

## 10. آموزش مدل

In [ ]:
# ============================================================
# CELL 10 — Train Qwen2.5-VL with LoRA
# ============================================================

import math
import torch
from pathlib import Path
from transformers import TrainingArguments, Trainer

# ------------------------------------------------------------
# Validate prerequisites
# ------------------------------------------------------------

assert "model" in globals(), "ابتدا مدل را بارگذاری کن."
assert "train_ds" in globals(), "ابتدا دیتاست آموزش را آماده کن."
assert "collate_fn" in globals(), "ابتدا collate_fn را تعریف کن."
assert "processor" in globals(), "ابتدا processor را بارگذاری کن."
assert "NUM_EPOCHS" in globals(), "NUM_EPOCHS تعریف نشده است."

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU فعال نیست. در Colab از Runtime > Change runtime type "
        "یک GPU انتخاب کن."
    )

if len(train_ds) == 0:
    raise ValueError("دیتاست آموزش خالی است.")

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
EPOCHS = NUM_EPOCHS

has_gpu = torch.cuda.is_available()
supports_bf16 = has_gpu and torch.cuda.is_bf16_supported()

# تعداد تقریبی گام‌های به‌روزرسانی در هر epoch
steps_per_epoch = math.ceil(
    len(train_ds) /
    (BATCH_SIZE * GRAD_ACCUMULATION)
)

total_training_steps = steps_per_epoch * EPOCHS

# در اجرای آزمایشی کوتاه، warmup اجباری نداشته باش.
warmup_steps = int(total_training_steps * 0.03)

print("Training samples:", len(train_ds))
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUMULATION)
print("Epochs:", EPOCHS)
print("Estimated steps per epoch:", steps_per_epoch)
print("Estimated total steps:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("BF16:", supports_bf16)

# ------------------------------------------------------------
# TrainingArguments
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./qwen25vl-lora-output",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,

    learning_rate=2e-4,
    warmup_steps=warmup_steps,

    logging_steps=5,

    save_strategy="epoch",
    save_total_limit=1,

    bf16=supports_bf16,
    fp16=has_gpu and not supports_bf16,

    optim="paged_adamw_8bit",

    gradient_checkpointing=True,
    remove_unused_columns=False,

    report_to="none",
    max_grad_norm=0.3,

    dataloader_num_workers=0,
)

# ------------------------------------------------------------
# Create Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

model.config.use_cache = False
torch.cuda.empty_cache()

print("\nStarting LoRA training...")

train_result = trainer.train()

print("\nTraining completed.")

# ------------------------------------------------------------
# Save adapter and processor
# ------------------------------------------------------------

model.config.use_cache = True

SAVE_DIR = "/content/qwen25vl-lora-final"

Path(SAVE_DIR).mkdir(
    parents=True,
    exist_ok=True,
)

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print("\nModel saved successfully.")
print("Save directory:", SAVE_DIR)
print("Training metrics:")

for key, value in train_result.metrics.items():
    print(f"{key}: {value}")

## 11. ارزیابی مدل fine-tuned

این ارزیابی فقط یک بار اجرا می‌شود؛ سلول تکراری ارزیابی حذف شده است.

In [ ]:
# ============================================================
# CELL 11 — Evaluate Fine-tuned Model
# ============================================================

import pandas as pd
from IPython.display import display

if "trainer" not in globals():
    raise RuntimeError(
        "Trainer پیدا نشد. ابتدا سلول آموزش را با موفقیت اجرا کن."
    )

if "test_ds" not in globals() or len(test_ds) == 0:
    raise RuntimeError("مجموعه تست موجود نیست یا خالی است.")

# استفاده از مدل آموزش‌دیده
finetuned_model = trainer.model
finetuned_model.config.use_cache = True
finetuned_model.eval()

# ارزیابی فقط یک بار
finetuned_metrics, finetuned_df = evaluate_model(
    finetuned_model,
    test_ds,
    "Fine-tuned model",
)

print("\nFine-tuned evaluation metrics:")
display(pd.DataFrame([finetuned_metrics]).style.format("{:.4f}"))

print("\nFine-tuned predictions:")
display(finetuned_df.head(10))

# بررسی تعداد نمونه‌های ارزیابی
print("\nEvaluation summary")
print("------------------")
print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))
print("Epochs:", NUM_EPOCHS)
print("Model:", MODEL_NAME)
print("LoRA enabled:", hasattr(finetuned_model, "peft_config"))
print("Evaluation failures:", finetuned_metrics["Evaluation failures"])

## 12. مقایسه واقعی، نمایش پیش‌بینی‌ها و ذخیره خروجی‌ها

این سلول معیارها را از خروجی‌های همین اجرا محاسبه می‌کند؛ هیچ عدد معیار به‌صورت دستی وارد نشده است. در پایان، فایل‌های ارزیابی و ZIP آداپتر ذخیره می‌شوند.

In [ ]:
# ============================================================
# FINAL CELL — Compare, Plot, Save Results and Zip Adapter
# ============================================================

from pathlib import Path
import json
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ------------------------------------------------------------
# 1. Verify evaluation outputs
# ------------------------------------------------------------

required_names = [
    "baseline_df",
    "finetuned_df",
    "baseline_metrics",
    "finetuned_metrics",
    "calculate_metrics",
]

missing = [
    name for name in required_names
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. "
        "Run the baseline and fine-tuned evaluation cells first."
    )

required_columns = {
    "index",
    "ground_truth",
    "prediction",
}

for name, df in [
    ("baseline_df", baseline_df),
    ("finetuned_df", finetuned_df),
]:
    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{name} is missing required columns: "
            f"{required_columns - set(df.columns)}"
        )

# Both models must be evaluated on the same test examples.
if baseline_df["index"].tolist() != finetuned_df["index"].tolist():
    raise ValueError(
        "Baseline and fine-tuned evaluations use different test indices."
    )

if (
    baseline_df["ground_truth"].astype(str).tolist()
    != finetuned_df["ground_truth"].astype(str).tolist()
):
    raise ValueError(
        "Ground-truth titles differ between the two evaluations."
    )

# ------------------------------------------------------------
# 2. Recalculate metrics from actual predictions
# ------------------------------------------------------------

baseline_failures = int(
    baseline_metrics.get("Evaluation failures", 0)
)

finetuned_failures = int(
    finetuned_metrics.get("Evaluation failures", 0)
)

baseline_metrics = calculate_metrics(
    baseline_df["prediction"].fillna("").astype(str).tolist(),
    baseline_df["ground_truth"].fillna("").astype(str).tolist(),
)
baseline_metrics["Evaluation failures"] = baseline_failures

finetuned_metrics = calculate_metrics(
    finetuned_df["prediction"].fillna("").astype(str).tolist(),
    finetuned_df["ground_truth"].fillna("").astype(str).tolist(),
)
finetuned_metrics["Evaluation failures"] = finetuned_failures

# ------------------------------------------------------------
# 3. Build the actual comparison table
# ------------------------------------------------------------

metric_names = [
    "Exact Match",
    "Levenshtein Ratio",
    "Word F1",
]

comparison = pd.DataFrame(
    {
        "Before fine-tuning": [
            baseline_metrics[name]
            for name in metric_names
        ],
        "After fine-tuning": [
            finetuned_metrics[name]
            for name in metric_names
        ],
    },
    index=metric_names,
)

comparison["Absolute change"] = (
    comparison["After fine-tuning"]
    - comparison["Before fine-tuning"]
)

print("Performance Comparison — actual evaluation")
display(comparison.style.format("{:.4f}"))

# ------------------------------------------------------------
# 4. Show predictions
# ------------------------------------------------------------

print("\nBaseline predictions:")
display(baseline_df.head(10))

print("\nFine-tuned predictions:")
display(finetuned_df.head(10))

# ------------------------------------------------------------
# 5. Save evaluation files
# ------------------------------------------------------------

results_dir = Path(RESULTS_DIR).resolve()
results_dir.mkdir(parents=True, exist_ok=True)

baseline_df.to_csv(
    results_dir / "baseline_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

finetuned_df.to_csv(
    results_dir / "finetuned_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison.to_csv(
    results_dir / "metrics_comparison.csv",
    index_label="Metric",
    encoding="utf-8-sig",
)

metadata = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "smoke_test": bool(SMOKE_TEST),
    "train_samples": len(train_ds),
    "test_samples": len(test_ds),
    "epochs": NUM_EPOCHS,
    "baseline_metrics": {
        key: float(value)
        for key, value in baseline_metrics.items()
    },
    "finetuned_metrics": {
        key: float(value)
        for key, value in finetuned_metrics.items()
    },
}

with open(
    results_dir / "metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )

# ------------------------------------------------------------
# 6. Plot and save comparison chart
# ------------------------------------------------------------

ax = comparison[
    ["Before fine-tuning", "After fine-tuning"]
].plot(
    kind="bar",
    figsize=(10, 5),
    width=0.7,
)

ax.set_title("Qwen2.5-VL: Before vs After Fine-tuning")
ax.set_ylabel("Score")
ax.set_xlabel("Evaluation metric")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="upper left")

plt.xticks(rotation=0)
plt.tight_layout()

plot_path = results_dir / "metrics_comparison.png"
plt.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 7. Zip the trained LoRA adapter
# ------------------------------------------------------------

adapter_dir = Path(SAVE_DIR).resolve()

if not adapter_dir.is_dir():
    raise FileNotFoundError(
        f"Trained adapter directory not found: {adapter_dir}"
    )

if not any(adapter_dir.iterdir()):
    raise ValueError(
        f"Adapter directory is empty: {adapter_dir}"
    )

zip_base = results_dir / "qwen25vl-bookcover-final"

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(adapter_dir),
)

print("\nAll tasks completed.")
print("Adapter directory:", adapter_dir)
print("Adapter ZIP:", zip_path)
print("Results directory:", results_dir)

print("\nSaved files:")
for path in sorted(results_dir.iterdir()):
    print("-", path.name)

## نکات اجرایی
- ابتدا با `SMOKE_TEST = True` اجرا کن. این فقط برای بررسی مسیر end-to-end است و برای ادعای علمی کافی نیست.
- برای ارزیابی اصلی، `SMOKE_TEST = False` کن و از ابتدای نوت‌بوک اجرا کن.
- نتایج قبل و بعد روی همان نمونه‌های تست محاسبه می‌شوند؛ اعداد نمودار دستی نیستند.
- آداپتر LoRA شامل وزن‌های کامل مدل پایه نیست؛ هنگام inference باید مدل پایه و آداپتر هر دو در دسترس باشند.
- اگر خطای Out of Memory دیدی، تعداد نمونه‌ها یا `max_pixels` را کاهش بده.